# Train and validate the XGBoost models

## 1. Annual mass-balance model

In [ ]:
from pathlib import Path
import itertools
import numpy as np
import pandas as pd

from os import environ

configured_root = environ.get("ML4GM_ROOT")
ROOT_DIR = Path(configured_root).expanduser().resolve() if configured_root else Path.cwd().resolve()
if not configured_root:
    if (ROOT_DIR / "01_Dataset.ipynb").is_file():
        ROOT_DIR = ROOT_DIR.parent
    if not (ROOT_DIR / "code" / "01_Dataset.ipynb").is_file():
        raise RuntimeError("Start this notebook from the release root or its code directory, or set ML4GM_ROOT.")
elif not ROOT_DIR.is_dir():
    raise FileNotFoundError(f"The configured ML4GM_ROOT directory does not exist: {ROOT_DIR}")
MODEL_DIR = ROOT_DIR / "models/annual"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"MODEL_DIR = {MODEL_DIR}")

In [ ]:
try:
    import xgboost as xgb
except ModuleNotFoundError:
    raise ModuleNotFoundError("Missing dependency `xgboost`. Install it before running this notebook.")

import dill
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler

with open(MODEL_DIR / "preprocessed_data.pkl", "rb") as fr:
    payload = dill.load(fr)

feature_columns = list(payload["feature_columns"])
X_all = pd.DataFrame(np.asarray(payload["X_all"], dtype=float), columns=feature_columns)
y_all = np.asarray(payload["y_all"], dtype=float)
rgiid_all = np.asarray(payload["rgiid_all"])
year_all = np.asarray(payload["year_all"])
o1region_all = np.asarray(payload.get("o1region_all", np.full(len(y_all), np.nan)))
o2region_all = np.asarray(payload.get("o2region_all", np.full(len(y_all), np.nan)))
feat_groups = payload.get("feat_groups", {})
area_all = np.asarray(payload["area_all"], dtype=float) if "area_all" in payload else X_all["Area"].to_numpy(dtype=float)
area_all = np.where(np.isfinite(area_all) & (area_all > 0), area_all, 1.0)

RANDOM_STATE = 42
SEARCH_FOLDS = 3
force_param_search = environ.get("ML4GM_FORCE_PARAM_SEARCH") == "1"
FORCE_PARAM_SEARCH_ANNUAL = force_param_search
FORCE_PARAM_SEARCH_WINTER = force_param_search
FORCE_PARAM_SEARCH_SUMMER = force_param_search

print(X_all.shape, y_all.shape)
print("glaciers =", pd.Series(rgiid_all).nunique())
print("years =", int(pd.Series(year_all).min()), int(pd.Series(year_all).max()))
print("feature groups =", feat_groups)

In [ ]:
def save_pkl(path, obj):
    with open(path, "wb") as fw:
        dill.dump(obj, fw)


def load_pkl(path):
    with open(path, "rb") as fr:
        return dill.load(fr)


INTEGER_PARAM_NAMES = {"n_estimators", "max_depth", "min_child_weight"}


def normalize_param_types(params):
    normalized = dict(params)
    for key in INTEGER_PARAM_NAMES:
        if key in normalized and pd.notna(normalized[key]):
            normalized[key] = int(normalized[key])
    return normalized


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[mask]
    y_pred = y_pred[mask]
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    denom = float(np.mean(np.abs(y_true))) if len(y_true) else np.nan
    if len(y_true) >= 2 and np.std(y_true) > 0 and np.std(y_pred) > 0:
        pearson_r = float(np.corrcoef(y_true, y_pred)[0, 1])
    else:
        pearson_r = np.nan
    return {
        "R2": float(r2_score(y_true, y_pred)),
        "RMSE": rmse,
        "MAE": mae,
        "Pearson_r": pearson_r,
        "NRMSE_mean_abs": float(rmse / denom) if np.isfinite(denom) and denom != 0 else np.nan,
    }

In [ ]:
params_path = MODEL_DIR / "xgboost_annual_unweighted_params.pkl"
search_results_path = MODEL_DIR / "xgboost_annual_unweighted_param_search.csv"

PARAM_GRID_SPEC = {
    "n_estimators": [200, 400, 600],
    "learning_rate": [0.03, 0.05],
    "max_depth": [3, 4, 5],
    "min_child_weight": [1, 5],
    "subsample": [0.7],
    "colsample_bytree": [0.6, 0.8],
    "reg_alpha": [0.0, 0.1],
    "reg_lambda": [1.0],
}
PARAM_GRID = [
    dict(zip(PARAM_GRID_SPEC.keys(), values))
    for values in itertools.product(*PARAM_GRID_SPEC.values())
]
print(f"Annual parameter candidates = {len(PARAM_GRID)}")

MODEL_BASE_PARAMS = {
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "tree_method": "hist",
}


def train_model(X_tr_df, y_tr, xgb_params):
    scaler = StandardScaler()
    Xf_tr = pd.DataFrame(
        scaler.fit_transform(X_tr_df),
        columns=feature_columns,
        index=X_tr_df.index,
    )
    model = xgb.XGBRegressor(**MODEL_BASE_PARAMS, **xgb_params)
    model.fit(Xf_tr, y_tr)
    return {"scaler": scaler, "model": model}


def predict_model(bundle, X_df):
    Xf = pd.DataFrame(
        bundle["scaler"].transform(X_df),
        columns=feature_columns,
        index=X_df.index,
    )
    return bundle["model"].predict(Xf)


def score_param_set(xgb_params):
    splitter = KFold(n_splits=min(SEARCH_FOLDS, len(X_all)), shuffle=True, random_state=RANDOM_STATE)
    oof = np.full(len(y_all), np.nan, dtype=float)

    for tr_idx, te_idx in splitter.split(X_all, y_all):
        X_tr_df = X_all.iloc[tr_idx].reset_index(drop=True)
        X_te_df = X_all.iloc[te_idx].reset_index(drop=True)
        y_tr = y_all[tr_idx]
        bundle = train_model(X_tr_df, y_tr, xgb_params)
        pred = predict_model(bundle, X_te_df)
        oof[te_idx] = pred

    mask = np.isfinite(oof)
    row = {**xgb_params, "n": int(mask.sum()), **compute_metrics(y_all[mask], oof[mask])}
    return row


if params_path.exists() and not FORCE_PARAM_SEARCH_ANNUAL:
    best_params = load_pkl(params_path)
    search_df = pd.read_csv(search_results_path) if search_results_path.exists() else pd.DataFrame()
else:
    search_rows = []
    for candidate_id, params in enumerate(PARAM_GRID, start=1):
        row = {"candidate_id": candidate_id, **score_param_set(params)}
        search_rows.append(row)
        print(row)
    search_df = pd.DataFrame(search_rows).sort_values(["RMSE", "R2"], ascending=[True, False]).reset_index(drop=True)
    best_params = search_df.iloc[0][list(PARAM_GRID_SPEC.keys())].to_dict()
    best_params = normalize_param_types(best_params)
    save_pkl(params_path, best_params)
    search_df.to_csv(search_results_path, index=False)

display(search_df.head(10) if len(search_df) else search_df)
print("best_params =", best_params)

In [ ]:
def fit_fold_model(X_tr_df, y_tr):
    return train_model(X_tr_df, y_tr, best_params)


def run_cv(splitter, split_name, groups=None):
    oof = np.full(len(y_all), np.nan, dtype=float)
    rows = []
    pred_rows = []

    split_iter = splitter.split(X_all, y_all, groups) if groups is not None else splitter.split(X_all, y_all)
    for fold_id, (tr_idx, te_idx) in enumerate(split_iter, start=1):
        X_tr_df = X_all.iloc[tr_idx].reset_index(drop=True)
        X_te_df = X_all.iloc[te_idx].reset_index(drop=True)
        y_tr = y_all[tr_idx]
        y_te = y_all[te_idx]
        bundle = fit_fold_model(X_tr_df, y_tr)
        pred = predict_model(bundle, X_te_df)
        oof[te_idx] = pred
        pred_rows.append(pd.DataFrame({
            "split": split_name,
            "fold": fold_id,
            "RGIId": rgiid_all[te_idx],
            "year": year_all[te_idx],
            "O1Region": o1region_all[te_idx],
            "O2Region": o2region_all[te_idx],
            "Area": area_all[te_idx],
            "obs": y_te,
            "pred": pred,
        }))

        row = {"split": split_name, "fold": fold_id, "n": int(len(te_idx))}
        row.update(compute_metrics(y_te, pred))
        if split_name == "logo_rgiid":
            row["held_rgiids"] = pd.Series(rgiid_all[te_idx]).nunique()
        if split_name == "loyo_year":
            row["held_years"] = ",".join(str(int(v)) for v in sorted(pd.Series(year_all[te_idx]).unique().tolist()))
        rows.append(row)
        print(row)

    detail_df = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)
    mask = np.isfinite(oof)
    summary_df = pd.DataFrame([{"split": split_name, "model": "xgboost_annual", "n": int(mask.sum()), **compute_metrics(y_all[mask], oof[mask])}])
    pred_df = pd.concat(pred_rows, ignore_index=True) if pred_rows else pd.DataFrame()
    return detail_df, summary_df, pred_df

In [ ]:
random_splitter = KFold(n_splits=min(10, len(X_all)), shuffle=True, random_state=RANDOM_STATE)
random_detail_df, random_summary_df, random_oof_df = run_cv(random_splitter, "random_kfold")
random_detail_df.to_csv(MODEL_DIR / "xgboost_annual_random_kfold_by_fold.csv", index=False)
random_oof_df.to_csv(MODEL_DIR / "xgboost_annual_random_kfold_oof_predictions.csv", index=False)
display(random_detail_df)
display(random_summary_df)

In [ ]:
global_bundle = train_model(X_all, y_all, best_params)
global_pred = predict_model(global_bundle, X_all)

save_pkl(
    MODEL_DIR / "xgboost_annual_global.pkl",
    {
        "feature_columns": feature_columns,
        "scaler": global_bundle["scaler"],
        "model": global_bundle["model"],
        "best_params": best_params,
        "sample_weighting": "none",
    },
)

global_summary_df = pd.DataFrame([
    {"split": "global_in_sample", "model": "xgboost_annual", "n": len(y_all), **compute_metrics(y_all, global_pred)}
])
global_summary_df.to_csv(MODEL_DIR / "xgboost_annual_global_in_sample.csv", index=False)
global_summary_df

In [ ]:
def booster_score(scores, feature, index):
    return float(scores.get(feature, scores.get(f"f{index}", 0.0)))


booster = global_bundle["model"].get_booster()
gain_scores = booster.get_score(importance_type="gain")
weight_scores = booster.get_score(importance_type="weight")
cover_scores = booster.get_score(importance_type="cover")
model_importance = getattr(global_bundle["model"], "feature_importances_", np.zeros(len(feature_columns), dtype=float))

fi_rows = []
for i, col in enumerate(feature_columns):
    fi_rows.append({
        "feature": col,
        "model_importance": float(model_importance[i]) if i < len(model_importance) else 0.0,
        "gain": booster_score(gain_scores, col, i),
        "weight": booster_score(weight_scores, col, i),
        "cover": booster_score(cover_scores, col, i),
    })

feature_importance_df = pd.DataFrame(fi_rows)
plot_metric = "gain"
if np.isclose(feature_importance_df["gain"].fillna(0).sum(), 0.0):
    print("Gain scores are unavailable from this booster; plotting model_importance instead.")
    plot_metric = "model_importance"
feature_importance_df = feature_importance_df.sort_values([plot_metric, "model_importance"], ascending=[False, False]).reset_index(drop=True)
feature_importance_df.to_csv(MODEL_DIR / "xgboost_annual_feature_importance.csv", index=False)
display(feature_importance_df.head(20))

In [ ]:
logo_detail_df, logo_summary_df, logo_oof_df = run_cv(LeaveOneGroupOut(), "logo_rgiid", groups=rgiid_all)
logo_detail_df.to_csv(MODEL_DIR / "xgboost_annual_logo_rgiid_by_fold.csv", index=False)
logo_oof_df.to_csv(MODEL_DIR / "xgboost_annual_logo_rgiid_oof_predictions.csv", index=False)

display(logo_summary_df)

loyo_detail_df, loyo_summary_df, loyo_oof_df = run_cv(LeaveOneGroupOut(), "loyo_year", groups=year_all)
loyo_detail_df.to_csv(MODEL_DIR / "xgboost_annual_loyo_year_by_fold.csv", index=False)
loyo_oof_df.to_csv(MODEL_DIR / "xgboost_annual_loyo_year_oof_predictions.csv", index=False)
display(loyo_detail_df)
display(loyo_summary_df)

In [ ]:
cv_summary_df = pd.concat([
    random_summary_df,
    logo_summary_df,
    loyo_summary_df,
    global_summary_df,
], ignore_index=True)
cv_summary_df.to_csv(MODEL_DIR / "xgboost_annual_validation_summary.csv", index=False)
cv_summary_df

## 2. Seasonal mass-balance models

In [ ]:
from pathlib import Path
import json
import math
import itertools
import dill
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


SEASONAL_DIR = ROOT_DIR / "data/processed/seasonal_training"
MODEL_DIR = ROOT_DIR / "models/annual_seasonal"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLIMATE_VARS = ['t2m', 'rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'sp', 'tp']
STATE_VARS = ['t2m', 'rsn', 'sde', 'tsn', 'fal', 'sp']
FLOW_VARS = ['sf', 'smlt', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'tp']
STATIC_FEATS = ['CenLon', 'CenLat', 'Area', 'Zmax', 'Zmed', 'Slope', 'Zmin', 'Lmax', 'Aspect']
SEASON_WINDOW_META = ['season_start_month', 'season_end_month']
RANDOM_STATE = 42


def normalize_region(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64')


def choose_mode(values):
    vals = [int(v) for v in values if pd.notna(v)]
    if not vals:
        return np.nan
    vc = pd.Series(vals).value_counts()
    top_count = vc.iloc[0]
    top_vals = sorted(int(v) for v in vc[vc == top_count].index.tolist())
    if len(top_vals) == 1:
        return top_vals[0]
    return int(round(float(np.median(top_vals))))


def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if valid.sum() == 0:
        return {
            'R2': np.nan,
            'RMSE': np.nan,
            'MAE': np.nan,
            'Pearson_r': np.nan,
            'NRMSE_mean_abs': np.nan,
        }
    y_true = y_true[valid]
    y_pred = y_pred[valid]
    if len(y_true) >= 2 and np.std(y_true) > 0:
        r2 = r2_score(y_true, y_pred)
        pearson = float(np.corrcoef(y_true, y_pred)[0, 1])
    else:
        r2 = np.nan
        pearson = np.nan
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    denom = float(np.mean(np.abs(y_true))) if len(y_true) else np.nan
    nrmse = rmse / denom if denom and np.isfinite(denom) and denom != 0 else np.nan
    return {
        'R2': r2,
        'RMSE': rmse,
        'MAE': mae,
        'Pearson_r': pearson,
        'NRMSE_mean_abs': nrmse,
    }


def build_gain_outputs(model, out_prefix, out_dir, feature_columns):
    booster = model.get_booster()
    gain_map = booster.get_score(importance_type='gain')
    gain_df = pd.DataFrame({
        'feature': feature_columns,
        'gain': [gain_map.get(f, gain_map.get(f'f{i}', 0.0)) for i, f in enumerate(feature_columns)]
    }).sort_values('gain', ascending=False)
    gain_csv = out_dir / f'{out_prefix}_feature_importance.csv'
    gain_df.to_csv(gain_csv, index=False)


def fit_predict_cv(X_df, y, rgiids, years, params):
    records = []
    all_random = []
    kf = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
    for fold, (tr_idx, te_idx) in enumerate(kf.split(X_df), start=1):
        model = XGBRegressor(
            objective='reg:squarederror',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            tree_method='hist',
            **params,
        )
        model.fit(X_df.iloc[tr_idx], y[tr_idx])
        pred = model.predict(X_df.iloc[te_idx])
        fold_df = pd.DataFrame({
            'fold': fold,
            'RGIId': rgiids[te_idx],
            'year': years[te_idx],
            'obs': y[te_idx],
            'pred': pred,
        })
        all_random.append(fold_df)
        m = compute_metrics(y[te_idx], pred)
        records.append({'split': 'random_kfold', 'fold': fold, **m})
    random_df = pd.concat(all_random, ignore_index=True)

    logo = LeaveOneGroupOut()
    logo_rows = []
    logo_records = []
    for fold, (tr_idx, te_idx) in enumerate(logo.split(X_df, y, groups=rgiids), start=1):
        model = XGBRegressor(
            objective='reg:squarederror',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            tree_method='hist',
            **params,
        )
        model.fit(X_df.iloc[tr_idx], y[tr_idx])
        pred = model.predict(X_df.iloc[te_idx])
        group = pd.DataFrame({
            'fold': fold,
            'RGIId': rgiids[te_idx],
            'year': years[te_idx],
            'obs': y[te_idx],
            'pred': pred,
        })
        logo_rows.append(group)
        m = compute_metrics(y[te_idx], pred)
        logo_records.append({'split': 'logo_rgiid', 'fold': fold, **m})
    logo_df = pd.concat(logo_rows, ignore_index=True)

    loyo = LeaveOneGroupOut()
    loyo_rows = []
    loyo_records = []
    for fold, (tr_idx, te_idx) in enumerate(loyo.split(X_df, y, groups=years), start=1):
        model = XGBRegressor(
            objective='reg:squarederror',
            n_jobs=-1,
            random_state=RANDOM_STATE,
            tree_method='hist',
            **params,
        )
        model.fit(X_df.iloc[tr_idx], y[tr_idx])
        pred = model.predict(X_df.iloc[te_idx])
        group = pd.DataFrame({
            'fold': fold,
            'RGIId': rgiids[te_idx],
            'year': years[te_idx],
            'obs': y[te_idx],
            'pred': pred,
        })
        loyo_rows.append(group)
        m = compute_metrics(y[te_idx], pred)
        loyo_records.append({'split': 'loyo_year', 'fold': fold, **m})
    loyo_df = pd.concat(loyo_rows, ignore_index=True)

    global_model = XGBRegressor(
        objective='reg:squarederror',
        n_jobs=-1,
        random_state=RANDOM_STATE,
        tree_method='hist',
        **params,
    )
    global_model.fit(X_df, y)
    in_sample_pred = global_model.predict(X_df)
    in_sample_metrics = compute_metrics(y, in_sample_pred)

    summary_rows = []
    for split_name, split_df in [('random_kfold', random_df), ('logo_rgiid', logo_df), ('loyo_year', loyo_df)]:
        m = compute_metrics(split_df['obs'], split_df['pred'])
        summary_rows.append({'split': split_name, **m})
    summary_rows.append({'split': 'global_in_sample', **in_sample_metrics})
    summary = pd.DataFrame(summary_rows)
    detail = pd.concat([
        pd.DataFrame(records),
        pd.DataFrame(logo_records),
        pd.DataFrame(loyo_records),
    ], ignore_index=True)
    return global_model, random_df, logo_df, loyo_df, detail, summary

### 2.1 Winter model

In [ ]:
from pathlib import Path
import json
import math
import itertools
import dill
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


SEASONAL_DIR = ROOT_DIR / "data/processed/seasonal_training"
MODEL_DIR = ROOT_DIR / "models/annual_seasonal"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLIMATE_VARS = ['t2m', 'rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'sp', 'tp']
STATE_VARS = ['t2m', 'rsn', 'sde', 'tsn', 'fal', 'sp']
FLOW_VARS = ['sf', 'smlt', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'tp']
STATIC_FEATS = ['CenLon', 'CenLat', 'Area', 'Zmax', 'Zmed', 'Slope', 'Zmin', 'Lmax', 'Aspect']
SEASON_WINDOW_META = ['season_start_month', 'season_end_month']
RANDOM_STATE = 42
INTEGER_PARAM_NAMES = ['n_estimators', 'max_depth', 'min_child_weight']


def normalize_param_types(params):
    normalized = dict(params)
    for key in INTEGER_PARAM_NAMES:
        if key in normalized and pd.notna(normalized[key]):
            normalized[key] = int(normalized[key])
    return normalized


def normalize_region(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64')


def choose_mode(values):
    vals = [int(v) for v in values if pd.notna(v)]
    if not vals:
        return np.nan
    vc = pd.Series(vals).value_counts()
    top_count = vc.iloc[0]
    top_vals = sorted(int(v) for v in vc[vc == top_count].index.tolist())
    if len(top_vals) == 1:
        return top_vals[0]
    return int(round(float(np.median(top_vals))))

In [ ]:
SEASON = 'winter'
PAYLOAD_PATH = MODEL_DIR / 'preprocessed_winter_data.pkl'
PARAM_SEARCH_PATH = MODEL_DIR / 'xgboost_winter_unweighted_param_search.csv'
PARAMS_PKL = MODEL_DIR / 'xgboost_winter_unweighted_params.pkl'
MODEL_PKL = MODEL_DIR / 'xgboost_winter_global.pkl'
VALIDATION_SUMMARY_CSV = MODEL_DIR / 'xgboost_winter_validation_summary.csv'
RANDOM_DETAIL_CSV = MODEL_DIR / 'xgboost_winter_random_kfold_by_fold.csv'
LOGO_DETAIL_CSV = MODEL_DIR / 'xgboost_winter_logo_rgiid_by_fold.csv'
LOYO_DETAIL_CSV = MODEL_DIR / 'xgboost_winter_loyo_year_by_fold.csv'
INSAMPLE_CSV = MODEL_DIR / 'xgboost_winter_global_in_sample.csv'

payload = dill.load(open(PAYLOAD_PATH, 'rb'))
feature_columns = list(payload['feature_columns'])
X_df = pd.DataFrame(payload['X_all'], columns=feature_columns).copy()
y = np.asarray(payload['y_all'], dtype=float)
rgiids = np.asarray(payload['rgiid_all'], dtype=object)
years = np.asarray(payload['year_all'], dtype=int)
print(SEASON, X_df.shape, len(np.unique(rgiids)))

In [ ]:
PARAM_GRID_SPEC = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05],
    'max_depth': [3, 4, 5],
    'min_child_weight': [1, 5],
    'subsample': [0.7],
    'colsample_bytree': [0.6, 0.8],
    'reg_alpha': [0.0, 0.1],
    'reg_lambda': [1.0],
}
PARAM_GRID = [
    dict(zip(PARAM_GRID_SPEC.keys(), values))
    for values in itertools.product(*PARAM_GRID_SPEC.values())
]
print(f'{SEASON.capitalize()} parameter candidates = {len(PARAM_GRID)}')

if PARAMS_PKL.exists() and not FORCE_PARAM_SEARCH_WINTER:
    best_params = normalize_param_types(dill.load(open(PARAMS_PKL, 'rb')))
    search_df = pd.read_csv(PARAM_SEARCH_PATH) if PARAM_SEARCH_PATH.exists() else pd.DataFrame()
else:
    search_rows = []
    cv = KFold(n_splits=SEARCH_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for candidate_id, params in enumerate(PARAM_GRID, start=1):
        oof = np.full(len(y), np.nan, dtype=float)
        for fold, (tr_idx, te_idx) in enumerate(cv.split(X_df), start=1):
            model = XGBRegressor(
                objective='reg:squarederror',
                n_jobs=-1,
                random_state=RANDOM_STATE,
                tree_method='hist',
                **params,
            )
            model.fit(X_df.iloc[tr_idx], y[tr_idx])
            oof[te_idx] = model.predict(X_df.iloc[te_idx])
        mask = np.isfinite(oof)
        row = {'candidate_id': candidate_id, **params, 'n': int(mask.sum()), **compute_metrics(y[mask], oof[mask])}
        search_rows.append(row)
        print(row)

    search_df = pd.DataFrame(search_rows).sort_values(['RMSE', 'R2'], ascending=[True, False]).reset_index(drop=True)
    best_params = normalize_param_types(search_df.iloc[0][list(PARAM_GRID_SPEC.keys())].to_dict())
    search_df.to_csv(PARAM_SEARCH_PATH, index=False)
    dill.dump(best_params, open(PARAMS_PKL, 'wb'))

display(search_df.head(10) if len(search_df) else search_df)
print('best_params =', best_params)

In [ ]:
best_params = normalize_param_types(dill.load(open(PARAMS_PKL, 'rb')))
dill.dump(best_params, open(PARAMS_PKL, 'wb'))
model, random_df, logo_df, loyo_df, detail_df, summary_df = fit_predict_cv(
    X_df=X_df,
    y=y,
    rgiids=rgiids,
    years=years,
    params=best_params,
)
model.get_booster().set_attr(sample_weighting='none')
dill.dump(model, open(MODEL_PKL, 'wb'))

random_df.to_csv(RANDOM_DETAIL_CSV, index=False)
logo_df.to_csv(LOGO_DETAIL_CSV, index=False)
loyo_df.to_csv(LOYO_DETAIL_CSV, index=False)
summary_df.to_csv(VALIDATION_SUMMARY_CSV, index=False)


insample_pred = model.predict(X_df)
pd.DataFrame({
    'obs': y,
    'pred': insample_pred,
}).to_csv(INSAMPLE_CSV, index=False)

build_gain_outputs(model, out_prefix=f'xgboost_{SEASON}', out_dir=MODEL_DIR, feature_columns=feature_columns)

summary_df

### 2.2 Summer model

In [ ]:
from pathlib import Path
import json
import math
import itertools
import dill
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    display = print
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.model_selection import KFold, LeaveOneGroupOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


SEASONAL_DIR = ROOT_DIR / "data/processed/seasonal_training"
MODEL_DIR = ROOT_DIR / "models/annual_seasonal"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLIMATE_VARS = ['t2m', 'rsn', 'sde', 'sf', 'smlt', 'tsn', 'fal', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'sp', 'tp']
STATE_VARS = ['t2m', 'rsn', 'sde', 'tsn', 'fal', 'sp']
FLOW_VARS = ['sf', 'smlt', 'slhf', 'ssr', 'strd', 'sshf', 'ssrd', 'tp']
STATIC_FEATS = ['CenLon', 'CenLat', 'Area', 'Zmax', 'Zmed', 'Slope', 'Zmin', 'Lmax', 'Aspect']
SEASON_WINDOW_META = ['season_start_month', 'season_end_month']
RANDOM_STATE = 42
INTEGER_PARAM_NAMES = ['n_estimators', 'max_depth', 'min_child_weight']


def normalize_param_types(params):
    normalized = dict(params)
    for key in INTEGER_PARAM_NAMES:
        if key in normalized and pd.notna(normalized[key]):
            normalized[key] = int(normalized[key])
    return normalized


def normalize_region(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64')


def choose_mode(values):
    vals = [int(v) for v in values if pd.notna(v)]
    if not vals:
        return np.nan
    vc = pd.Series(vals).value_counts()
    top_count = vc.iloc[0]
    top_vals = sorted(int(v) for v in vc[vc == top_count].index.tolist())
    if len(top_vals) == 1:
        return top_vals[0]
    return int(round(float(np.median(top_vals))))

In [ ]:
SEASON = 'summer'
PAYLOAD_PATH = MODEL_DIR / 'preprocessed_summer_data.pkl'
PARAM_SEARCH_PATH = MODEL_DIR / 'xgboost_summer_unweighted_param_search.csv'
PARAMS_PKL = MODEL_DIR / 'xgboost_summer_unweighted_params.pkl'
MODEL_PKL = MODEL_DIR / 'xgboost_summer_global.pkl'
VALIDATION_SUMMARY_CSV = MODEL_DIR / 'xgboost_summer_validation_summary.csv'
RANDOM_DETAIL_CSV = MODEL_DIR / 'xgboost_summer_random_kfold_by_fold.csv'
LOGO_DETAIL_CSV = MODEL_DIR / 'xgboost_summer_logo_rgiid_by_fold.csv'
LOYO_DETAIL_CSV = MODEL_DIR / 'xgboost_summer_loyo_year_by_fold.csv'
INSAMPLE_CSV = MODEL_DIR / 'xgboost_summer_global_in_sample.csv'

payload = dill.load(open(PAYLOAD_PATH, 'rb'))
feature_columns = list(payload['feature_columns'])
X_df = pd.DataFrame(payload['X_all'], columns=feature_columns).copy()
y = np.asarray(payload['y_all'], dtype=float)
rgiids = np.asarray(payload['rgiid_all'], dtype=object)
years = np.asarray(payload['year_all'], dtype=int)
print(SEASON, X_df.shape, len(np.unique(rgiids)))

In [ ]:
PARAM_GRID_SPEC = {
    'n_estimators': [200, 400, 600],
    'learning_rate': [0.03, 0.05],
    'max_depth': [3, 4, 5],
    'min_child_weight': [1, 5],
    'subsample': [0.7],
    'colsample_bytree': [0.6, 0.8],
    'reg_alpha': [0.0, 0.1],
    'reg_lambda': [1.0],
}
PARAM_GRID = [
    dict(zip(PARAM_GRID_SPEC.keys(), values))
    for values in itertools.product(*PARAM_GRID_SPEC.values())
]
print(f'{SEASON.capitalize()} parameter candidates = {len(PARAM_GRID)}')

if PARAMS_PKL.exists() and not FORCE_PARAM_SEARCH_SUMMER:
    best_params = normalize_param_types(dill.load(open(PARAMS_PKL, 'rb')))
    search_df = pd.read_csv(PARAM_SEARCH_PATH) if PARAM_SEARCH_PATH.exists() else pd.DataFrame()
else:
    search_rows = []
    cv = KFold(n_splits=SEARCH_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    for candidate_id, params in enumerate(PARAM_GRID, start=1):
        oof = np.full(len(y), np.nan, dtype=float)
        for fold, (tr_idx, te_idx) in enumerate(cv.split(X_df), start=1):
            model = XGBRegressor(
                objective='reg:squarederror',
                n_jobs=-1,
                random_state=RANDOM_STATE,
                tree_method='hist',
                **params,
            )
            model.fit(X_df.iloc[tr_idx], y[tr_idx])
            oof[te_idx] = model.predict(X_df.iloc[te_idx])
        mask = np.isfinite(oof)
        row = {'candidate_id': candidate_id, **params, 'n': int(mask.sum()), **compute_metrics(y[mask], oof[mask])}
        search_rows.append(row)
        print(row)

    search_df = pd.DataFrame(search_rows).sort_values(['RMSE', 'R2'], ascending=[True, False]).reset_index(drop=True)
    best_params = normalize_param_types(search_df.iloc[0][list(PARAM_GRID_SPEC.keys())].to_dict())
    search_df.to_csv(PARAM_SEARCH_PATH, index=False)
    dill.dump(best_params, open(PARAMS_PKL, 'wb'))

display(search_df.head(10) if len(search_df) else search_df)
print('best_params =', best_params)

In [ ]:
best_params = normalize_param_types(dill.load(open(PARAMS_PKL, 'rb')))
dill.dump(best_params, open(PARAMS_PKL, 'wb'))
model, random_df, logo_df, loyo_df, detail_df, summary_df = fit_predict_cv(
    X_df=X_df,
    y=y,
    rgiids=rgiids,
    years=years,
    params=best_params,
)
model.get_booster().set_attr(sample_weighting='none')
dill.dump(model, open(MODEL_PKL, 'wb'))

random_df.to_csv(RANDOM_DETAIL_CSV, index=False)
logo_df.to_csv(LOGO_DETAIL_CSV, index=False)
loyo_df.to_csv(LOYO_DETAIL_CSV, index=False)
summary_df.to_csv(VALIDATION_SUMMARY_CSV, index=False)


insample_pred = model.predict(X_df)
pd.DataFrame({
    'obs': y,
    'pred': insample_pred,
}).to_csv(INSAMPLE_CSV, index=False)

build_gain_outputs(model, out_prefix=f'xgboost_{SEASON}', out_dir=MODEL_DIR, feature_columns=feature_columns)

summary_df